Updated model to modify for normalization N and baseline B.

In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from src.RealDataReplication import RealDataReplicationDeconvolution
from src.rg1_refactor import load_default_chrom_configs

config, _ = load_default_chrom_configs()

real_deconv1 = RealDataReplicationDeconvolution(config, replicate=1)

Initializing N using config H.
Initializing B using timepoint 0


###### 

In [3]:
real_deconv1.config.update_timepoints()
real_deconv1.config.calculate_H()
real_deconv1.config.params_dic

{'mu0': -26.7901,
 'delta': 8.1805,
 'sigma0': 10.9885,
 'sigmav': 0.1424,
 'lambda': 68.1914,
 'gamma1': 0.0041,
 'gamma2': 0.2547,
 'halted': 0.2245,
 'alpha': 22}

In [4]:
from src.optimize_H import ParameterOptimizer

In [5]:
timepoints = config.timepoints
initial_values = config.params_dic

init_params_df = pd.DataFrame(initial_values, index=['value']).T
init_params_df = init_params_df.drop('gamma1')
init_params_df = init_params_df.drop('gamma2')
init_params_df = init_params_df.drop('alpha')
init_params_df

,value
mu0,-26.7901
delta,8.1805
sigma0,10.9885
sigmav,0.1424
lambda,68.1914
halted,0.2245


In [6]:
bounds_dic = {
    'mu0': (-30, -10),
    'lambda': (60, 80),
    'delta': (1, 14),
    'sigma0': (1, 14),
    'sigmav': (0.01, 1.),
    'halted': (0.0, 1.),
}

bounds_params_df = pd.DataFrame(bounds_dic, index=['min', 'max']).T
params_df = init_params_df.join(bounds_params_df)
params_df


,value,min,max
mu0,-26.7901,-30.00,-10.0
delta,8.1805,1.00,14.0
sigma0,10.9885,1.00,14.0
sigmav,0.1424,0.01,1.0
lambda,68.1914,60.00,80.0
halted,0.2245,0.00,1.0


In [7]:
real_deconv1.setup_deconvolution(real_deconv1.config)

Initializing N using config H.
Initializing B using timepoint 0


In [8]:
real_deconv1.iterative_deconvolution_updates(5)

Iteration 0
0/368 - 00:00:00.002
100/368 - 00:00:00.093
200/368 - 00:00:00.181
300/368 - 00:00:00.262
0.006283419901301835
Iteration completed 00:00:00.329, rn=0.006283419901301833
Iteration 1
0/368 - 00:00:00.331
100/368 - 00:00:00.412
200/368 - 00:00:00.496
300/368 - 00:00:00.577
0.00442703733302984
Iteration completed 00:00:00.644, rn=0.00442703733302984
Iteration 2
0/368 - 00:00:00.645
100/368 - 00:00:00.726
200/368 - 00:00:00.807
300/368 - 00:00:00.891
0.004113152765615543
Iteration completed 00:00:00.995, rn=0.0041131527656155424
Iteration 3
0/368 - 00:00:00.996
100/368 - 00:00:01.077
200/368 - 00:00:01.157
300/368 - 00:00:01.237
0.003953469425160773
Iteration completed 00:00:01.304, rn=0.003953469425160774
Iteration 4
0/368 - 00:00:01.305
100/368 - 00:00:01.384
200/368 - 00:00:01.465
300/368 - 00:00:01.545
0.003885066912221466
Iteration completed 00:00:01.619, rn=0.0038850669122214667


In [9]:
optimizer = ParameterOptimizer(
    init_params_df=params_df,
    config=config,
    N=real_deconv1.N,
    F=real_deconv1.F,
    B=real_deconv1.B,
    G=real_deconv1.G
)


In [ ]:
from src.timer import Timer
from src.single_G1_config import calcH

timer = Timer()

num_epochs = 200
num_iterations_N_B = 5
update_params_df = pd.DataFrame()

N = real_deconv1.N
B = real_deconv1.B

optimizer = ParameterOptimizer(
    init_params_df=params_df,
    config=config,
    N=real_deconv1.N,
    F=real_deconv1.F,
    B=real_deconv1.B,
    G=real_deconv1.G
)

for epoch in range(num_epochs):
    print("Epoch: ", epoch)
    
    optimizer.optimize(maxiter=1000, verbose=True)

    real_deconv1.H = optimizer.current_H
        
    real_deconv1.iterative_deconvolution_updates(
        total_iterations=num_iterations_N_B, timer=timer,
        initial_B=B, initial_N=N)
    timer.print_time()
    
    params_row = pd.DataFrame([optimizer.params_df['value']], index=[epoch])
    params_row['opt_H_loss'] = optimizer.rn
    params_row['F_rn'] = real_deconv1.rn

    update_params_df = pd.concat([update_params_df, params_row])
    
    optimizer.N = real_deconv1.N
    optimizer.B = real_deconv1.B
    optimizer.F = real_deconv1.F
    
    N = real_deconv1.N
    B = real_deconv1.B

    print(update_params_df.iloc[-1])


Epoch:  0
Optimization [1]: Current loss: 0.003885066912221466
Optimization [101]: Current loss: 0.0037720513090949686
Optimization [201]: Current loss: 0.0037490091186672698


In [ ]:
r, c = 2, 5

plt.figure(figsize=(16, 6))

for i, param in enumerate(update_params_df.columns):
        
    plt.subplot(r, c, i+1)
    plt.plot(update_params_df[param])
    plt.title(param)
    
plt.subplots_adjust(hspace=0.5, wspace=0.35)
plt.suptitle(f"Cell cycle parameter updates, {num_epochs} epochs")

In [ ]:
plt.imshow(real_deconv1.H, vmax=0.05)

In [ ]:
real_deconv1.plot_heatmaps()

In [ ]:
plt.imshow(config.H, vmax=0.05)

In [ ]:
config.params_dic